# W07 -- Data Summary: Target Definition, Feature Engineering, and Modeling

**Major revision:** earlier versions of this notebook predicted `is_unaffordable` (a *persistent state*, ~13% base rate, flips only 3.9% of the time per year). A trivial "assume nothing changed since last year" baseline scored accuracy 0.961 / F1 0.852 on that target -- **higher** than the fitted model -- because the label barely ever changes. That made every reported accuracy/F1 number on this notebook misleading, not because of a coding bug (a real one was found and fixed too, see below), but because the target itself made the problem too easy.

This version predicts **`collapse_onset`** instead -- the actual transition event (the first quarter a metro crosses into unaffordable), restricted to the "at-risk" population (metros not already unaffordable). This is a genuinely hard target: a naive "always predict no collapse" baseline gets F1 = 0.0, and a persistence baseline gets F1 = 0.03. Any real score here reflects real signal.

Two more changes that follow directly from this:
1. **Training population selection changed** from "20 cities picked for geographic spread" to "every city with complete feature data that has at least one real `collapse_onset` event" -- the old geographic selection gave only 3 usable positive examples (the original anchor cities); the new one gives ~85+ across ~55+ cities.
2. **A real data-leakage bug was found and fixed**: the train/validation split used to be row-level random (`train_test_split`), which let the same city appear on both sides -- confirmed empirically (19/19 overlapping cities, 100%). Fixed with `StratifiedGroupKFold`, which keeps every city on one side of any split while still balancing classes.

**Input:** `data/final_data/price_changes_with_collapse_flags.csv`.

**Outputs:** train/val/holdout CSV splits, metrics tables (now including PR-AUC and an explicit comparison to the no-skill baseline), and SHAP/risk-ranking figures under `output/`.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
import optuna

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedGroupKFold, GroupKFold,
    cross_val_score, cross_val_predict
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, classification_report,
    make_scorer, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.discrete.discrete_model import Logit

In [2]:
# Load the R-joined panel (CBSA is the join key across all 5 data sources)
df = pd.read_csv("../../data/final_data/price_changes_with_collapse_flags.csv")

# R exports columns like "metro_name.x" -- flatten dots to underscores for easier access
df.columns = df.columns.str.replace('.', '_', regex=False)

print(df.shape)
print(df.dtypes[['cbsa', 'metro_name_x', 'year', 'qtr']])

(83639, 33)
cbsa            int64
metro_name_x      str
year            int64
qtr             int64
dtype: object


/var/folders/8w/0g4j4t9j4jz3cfq3qwhrsn2r0000gn/T/ipykernel_45743/4133038028.py:2: DtypeWarning: Columns (0: RegionName.y.y) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/final_data/price_changes_with_collapse_flags.csv")


## Target definition: affordability collapse onset (hazard framing)

`collapse_onset` marks the first quarter a metro's price-to-income ratio crosses above a threshold and stays there. We use **5.0** as that threshold -- chosen below by comparing onset dates at 4.0 / 4.5 / 5.0 / 5.5 and picking the value that gives the tightest, most realistic cluster of onset dates for Austin, Boise, and Tampa (the three original case-study cities; they remain useful reference points even though the training population below is no longer built around just them).

The modeling population is further restricted to **at-risk rows** (`prev_unaffordable == False`) -- once a metro is already unaffordable, "predicting an onset" for it doesn't mean anything; the question is only meaningful for metros that are currently still affordable.

In [3]:
# Confirms Austin (12420), Boise (14260), and Tampa (45294) all have data,
# and that the onset dates line up with what we expect: Austin 2021Q2, Boise 2019Q3, Tampa 2021Q4.
ANCHOR_CITIES = {'Austin': 12420.0, 'Boise': 14260.0, 'Tampa': 45294.0}

df = df.sort_values(['cbsa', 'year', 'qtr']).reset_index(drop=True)
g = df.groupby('cbsa')

df['is_unaffordable'] = df['price_to_income_ratio'] > 5.0
df['prev_unaffordable'] = g['is_unaffordable'].shift(1).fillna(False).astype(bool)
df['collapse_onset'] = df['is_unaffordable'] & (~df['prev_unaffordable'])

first_collapse = (
    df[df['collapse_onset']]
    .groupby('cbsa')[['metro_name_x', 'year', 'qtr', 'price_to_income_ratio']]
    .first()
)
print("Collapse onset check (anchor cities):")
print(first_collapse.loc[first_collapse.index.isin(ANCHOR_CITIES.values())])

Collapse onset check (anchor cities):
                           metro_name_x  year  qtr  price_to_income_ratio
cbsa                                                                     
12420  Austin-Round Rock-San Marcos, TX  2021    2               5.194781
14260                    Boise City, ID  2019    3               5.046829
45294                  Tampa, FL (MSAD)  2021    4               5.213635


In [4]:
# Sensitivity check: does the onset date move much if the threshold isn't 5.0?
for threshold in [4.0, 4.5, 5.0, 5.5]:
    print(f"\n--- Threshold: {threshold} ---")
    for city, code_ in ANCHOR_CITIES.items():
        sub = df[df['cbsa'] == code_].sort_values(['year', 'qtr']).copy()
        sub = sub[sub['price_to_income_ratio'].notna()]
        sub['is_unaffordable'] = sub['price_to_income_ratio'] > threshold
        sub['prev_unaffordable'] = sub['is_unaffordable'].shift(1).fillna(False)
        sub['onset'] = sub['is_unaffordable'] & ~sub['prev_unaffordable']
        onset_row = sub[sub['onset']].head(1)
        if len(onset_row) > 0:
            print(f"  {city}: {onset_row['year'].values[0]}Q{onset_row['qtr'].values[0]}")
        else:
            print(f"  {city}: never crosses this threshold")

# Result: 5.0 gives the tightest, most realistic cluster of onset dates.


--- Threshold: 4.0 ---
  Austin: 2014Q3
  Boise: 2016Q2
  Tampa: 2017Q4

--- Threshold: 4.5 ---
  Austin: 2020Q4
  Boise: 2017Q4
  Tampa: 2021Q2

--- Threshold: 5.0 ---
  Austin: 2021Q2
  Boise: 2019Q3
  Tampa: 2021Q4

--- Threshold: 5.5 ---
  Austin: 2021Q3
  Boise: 2020Q4
  Tampa: 2022Q2


## Feature engineering

Every feature below is lagged by 4 quarters (1 year) before any rolling calculation, so nothing accidentally sees the same-quarter data used to build `collapse_onset`. All 13 features are used by both models.

* Price-to-income level and 5-year change
* ZHVI momentum: YoY, QoQ, and a 3-year rolling trend of YoY
* HPI momentum: YoY and a properly lagged 3-year change
* Population velocity and acceleration
* Rent growth (ZORI YoY)
* Local unemployment rate (level)
* For-sale inventory (QoQ change)
* S&P 500 (YoY change) -- the only feature identical across every metro in a given quarter

Unemployment, inventory, and the S&P 500 return are left without a monotonic direction assumption (see the Modeling section for why); every other feature is assumed to move in the same direction as risk.

**Rule:** never let a feature use current- or future-period information that overlaps with label construction.

In [5]:
LAG = 4

df = df.sort_values(['cbsa', 'year', 'qtr']).reset_index(drop=True)
g = df.groupby('cbsa')

# Price-to-income level and 5-year change
df['price_to_income_lag'] = g['price_to_income_ratio'].shift(LAG)
df['price_to_income_5yr_chg'] = df.groupby('cbsa')['price_to_income_lag'].transform(
    lambda x: x - x.shift(20)
)

# ZHVI momentum -- YoY, QoQ, and a 3yr rolling slope of lagged YoY
df['zhvi_yoy_fixed'] = df.groupby('cbsa')['zhvi_qtr'].transform(lambda x: x.pct_change(4))
df['zhvi_qoq_fixed'] = df.groupby('cbsa')['zhvi_qtr'].transform(lambda x: x.pct_change(1))
df['zhvi_yoy_lag'] = df.groupby('cbsa')['zhvi_yoy_fixed'].shift(LAG)
df['zhvi_qoq_lag'] = df.groupby('cbsa')['zhvi_qoq_fixed'].shift(LAG)
df['three-year_home_price_growth_trend'] = df.groupby('cbsa')['zhvi_yoy_lag'].transform(
    lambda x: x.rolling(12, min_periods=12).apply(
        lambda y: np.polyfit(np.arange(len(y)), y, 1)[0] if y.notna().all() else np.nan
    )
)

# HPI momentum -- YoY (already lagged) and a lagged 3yr change
df['hpi_yoy'] = df.groupby('cbsa')['index_sa'].transform(lambda x: x.pct_change(4))
df['hpi_yoy_lag'] = df.groupby('cbsa')['hpi_yoy'].shift(LAG)
df['hpi_3yr_chg'] = g['index_sa'].transform(lambda x: (x / x.shift(12)) - 1) * 100  # contemporaneous, NOT used as a feature -- reference only
df['hpi_3yr_chg_lag'] = df.groupby('cbsa')['hpi_3yr_chg'].shift(LAG)

# Population velocity and acceleration, lagged
df["pop_yoy_fixed"] = g["population"].transform(lambda x: x.pct_change(4))
df["pop_velocity_fixed"] = df.groupby("cbsa")["pop_yoy_fixed"].diff()
df["pop_velocity_lag"] = df.groupby("cbsa")["pop_velocity_fixed"].shift(LAG)
df["pop_acceleration_fixed"] = df.groupby("cbsa")["pop_velocity_fixed"].diff()
df["pop_acceleration_lag"] = df.groupby("cbsa")["pop_acceleration_fixed"].shift(LAG)

# Rent growth (ZORI YoY), lagged
df["zori_yoy_fixed"] = df.groupby("cbsa")["zori_qtr"].transform(lambda x: x.pct_change(4))
df["zori_yoy_lag"] = df.groupby("cbsa")["zori_yoy_fixed"].shift(LAG)

# Local labor market: unemployment rate level, lagged. Direction left unconstrained.
df["unemployment_rate_lag"] = df.groupby("cbsa")["unemployment_rate"].shift(LAG)

# Housing supply: for-sale inventory, QoQ change (not YoY -- see coverage note in
# the R notebook). Direction left unconstrained.
df["inv_qoq_fixed"] = df.groupby("cbsa")["inventory_qtr"].transform(lambda x: x.pct_change(1))
df["inv_qoq_lag"] = df.groupby("cbsa")["inv_qoq_fixed"].shift(LAG)

# National S&P 500, YoY change, lagged. Direction left unconstrained.
df["sp500_yoy_fixed"] = df.groupby("cbsa")["sp500_qtr"].transform(lambda x: x.pct_change(4))
df["sp500_yoy_lag"] = df.groupby("cbsa")["sp500_yoy_fixed"].shift(LAG)

ALL_FEATURES = [
    "price_to_income_lag",
    "price_to_income_5yr_chg",
    "zhvi_yoy_lag",
    "zhvi_qoq_lag",
    "three-year_home_price_growth_trend",
    "hpi_yoy_lag",
    "hpi_3yr_chg_lag",
    "pop_velocity_lag",
    "pop_acceleration_lag",
    "zori_yoy_lag",
    "unemployment_rate_lag",
    "inv_qoq_lag",
    "sp500_yoy_lag",
]

## Selecting the training population: cities with real events, not geographic spread

The previous approach picked 20 cities to spread across US regions/states. That's the wrong criterion for this target: a "safe" city that never crosses the affordability threshold contributes zero usable positive examples no matter how well it represents its region. What actually matters is having enough real `collapse_onset` events to learn from.

**New rule:** restrict to at-risk rows (not already unaffordable), then train on every city that has complete feature data **and** at least one real `collapse_onset` event anywhere in its history. This is the direct, necessary consequence of switching to the honest target -- not scope creep.

In [6]:
at_risk = df[~df["prev_unaffordable"]].dropna(subset=ALL_FEATURES + ["collapse_onset"]).copy()
event_cities = at_risk.loc[at_risk["collapse_onset"], "cbsa"].unique()

print("At-risk rows with complete features:", len(at_risk))
print("Distinct cities with >=1 real collapse_onset event (complete features):", len(event_cities))
print("Total collapse_onset=True rows in this population:", int(at_risk["collapse_onset"].sum()))

training_cbsa_map = (
    at_risk[at_risk["cbsa"].isin(event_cities)]
    .groupby("cbsa")["metro_name_x"].first()
    .reset_index()
    .set_index("metro_name_x")["cbsa"]
    .to_dict()
)
print(f"\nTraining cities: {len(training_cbsa_map)}")
print("Anchor cities included:", all(c in training_cbsa_map.values() for c in ANCHOR_CITIES.values()))

training_pool_all = at_risk[at_risk["cbsa"].isin(training_cbsa_map.values())].copy()
print(f"\nTraining pool: {len(training_pool_all)} rows, positive rate {training_pool_all['collapse_onset'].mean():.1%}")

# Context only, not a selection criterion: rough geographic spread of the event cities.
state = training_pool_all.groupby("cbsa")["metro_name_x"].first().str.extract(r",\s*([A-Z]{2})")[0]
print("Distinct states represented among training cities:", state.nunique())

At-risk rows with complete features: 6089
Distinct cities with >=1 real collapse_onset event (complete features): 57
Total collapse_onset=True rows in this population: 86

Training cities: 57
Anchor cities included: False

Training pool: 876 rows, positive rate 9.8%
Distinct states represented among training cities: 24


## Findings: why this framing, and what the honest baselines are

Two things worth stating plainly before looking at any model output:

1. **`is_unaffordable` (the old target) is dominated by persistence.** The label only flips 3.9% of the time over any 4-quarter window. A trivial rule (`price_to_income_lag > 5.0`, i.e. "was it already true a year ago") scores accuracy 0.961 / F1 0.852 on it -- higher than the fitted model achieved. That's why this notebook switched targets rather than just re-tuning.
2. **`collapse_onset` (the new target) cannot be gamed the same way.** In the at-risk population, a persistence baseline scores F1 = 0.03 and an "always predict no collapse" baseline scores F1 = 0.0. Any real F1/PR-AUC number below is measuring something real.

In [7]:
# Create output directory if it doesn't exist
os.makedirs("output", exist_ok=True)

target = "collapse_onset"
model1_pool = training_pool_all.copy()

X_all = model1_pool[ALL_FEATURES]
y_all = model1_pool[target].astype(int)
groups_m1 = model1_pool["cbsa"]

model1_pool.to_csv("output/model1_pool.csv", index=False)
print("Model 1 pool:", len(model1_pool), "rows,", groups_m1.nunique(), "cities,",
      f"{y_all.mean():.1%} positive")

Model 1 pool: 876 rows, 57 cities, 9.8% positive


In [8]:
# Model 2 uses the same event-city population as Model 1 -- see the Modeling
# section for how the two differ procedurally.
model2_train_cities = dict(training_cbsa_map)
model2_pool = training_pool_all.copy()

Xb_all = model2_pool[ALL_FEATURES]
yb_all = model2_pool[target].astype(int)
groups_m2 = model2_pool["cbsa"]

model2_pool.to_csv("output/model2_pool.csv", index=False)
print("Model 2 pool:", len(model2_pool), "rows,", groups_m2.nunique(), "cities,",
      f"{yb_all.mean():.1%} positive")

Model 2 pool: 876 rows, 57 cities, 9.8% positive


In [9]:
# Holdout score set: at-risk metros NOT in the training population, complete
# features. These are the metros actually being ranked for early-warning risk.
holdout_scoring = at_risk[~at_risk["cbsa"].isin(training_cbsa_map.values())].dropna(
    subset=ALL_FEATURES
).copy()

print("Holdout scoring rows:", len(holdout_scoring), "| cities:", holdout_scoring["cbsa"].nunique())

holdout_scoring.to_csv("output/holdout_scoring.csv", index=False)
print("Saved: model1_pool.csv, model2_pool.csv, holdout_scoring.csv")

Holdout scoring rows: 5213 | cities: 296
Saved: model1_pool.csv, model2_pool.csv, holdout_scoring.csv


## Modeling: XGBoost + SHAP explainability

With a ~9-10% positive rate (much better than the <1% unconditional rate, but still imbalanced), both models use `scale_pos_weight` to avoid collapsing to "always predict negative." **PR-AUC (average precision) is the primary reported metric**, always shown next to the no-skill baseline (the positive rate itself) -- accuracy alone is not meaningful at this class balance.

In [10]:
os.makedirs("output/figures", exist_ok=True)
os.makedirs("output/tables", exist_ok=True)

ALL_FEATURES = [
    "price_to_income_lag",
    "price_to_income_5yr_chg",
    "zhvi_yoy_lag",
    "zhvi_qoq_lag",
    "three-year_home_price_growth_trend",
    "hpi_yoy_lag",
    "hpi_3yr_chg_lag",
    "pop_velocity_lag",
    "pop_acceleration_lag",
    "zori_yoy_lag",
    "unemployment_rate_lag",
    "inv_qoq_lag",
    "sp500_yoy_lag",
]

UNCONSTRAINED_FEATURES = {"unemployment_rate_lag", "inv_qoq_lag", "sp500_yoy_lag"}
MONOTONE_INCREASING = tuple(0 if f in UNCONSTRAINED_FEATURES else 1 for f in ALL_FEATURES)

In [11]:
# Reload from the saved splits so this modeling section can be re-run independently
# of the feature-engineering cells above.
model1_pool = pd.read_csv("output/model1_pool.csv")
model2_pool = pd.read_csv("output/model2_pool.csv")
holdout_scoring = pd.read_csv("output/holdout_scoring.csv")

target = "collapse_onset"
X_all = model1_pool[ALL_FEATURES]
y_all = model1_pool[target].astype(int)
groups_m1 = model1_pool["cbsa"]

Xb_all = model2_pool[ALL_FEATURES]
yb_all = model2_pool[target].astype(int)
groups_m2 = model2_pool["cbsa"]

### Model 1: Early-Warning Indicator Model

Model 1 identifies which of the engineered indicators is most associated with a real `collapse_onset` event across every city with complete feature data that has ever had one -- not just Austin/Boise/Tampa. Default hyperparameters, used for SHAP explainability.

In [12]:
# Grouped split (StratifiedGroupKFold): no city appears on both sides. This is
# the fix for the leakage bug found earlier (plain train_test_split let the
# same city appear in both train and val -- confirmed empirically at 19/19
# overlap on the old is_unaffordable setup).
neg1, pos1 = (y_all == 0).sum(), (y_all == 1).sum()
scale_pos_weight_1 = neg1 / pos1
print(f"Model 1 class balance: neg={neg1}, pos={pos1}, scale_pos_weight={scale_pos_weight_1:.1f}")

split_m1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx_m1, val_idx_m1 = next(split_m1.split(X_all, y_all, groups=groups_m1))
Xa_train, Xa_val = X_all.iloc[train_idx_m1], X_all.iloc[val_idx_m1]
ya_train, ya_val = y_all.iloc[train_idx_m1], y_all.iloc[val_idx_m1]

train_cities_m1 = set(groups_m1.iloc[train_idx_m1])
val_cities_m1 = set(groups_m1.iloc[val_idx_m1])
print("City overlap between train and val (should be 0):", len(train_cities_m1 & val_cities_m1))
print(f"Train: {len(Xa_train)} rows / {len(train_cities_m1)} cities, "
      f"Val: {len(Xa_val)} rows / {len(val_cities_m1)} cities, "
      f"val positive rate: {ya_val.mean():.1%}")

model1_fresh = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight_1,
    monotone_constraints=MONOTONE_INCREASING,
    random_state=42
)
model1_fresh.fit(Xa_train, ya_train)

pred_fresh = model1_fresh.predict(Xa_val)
proba_fresh = model1_fresh.predict_proba(Xa_val)[:, 1]

baseline_prauc_1 = ya_val.mean()
print("\nFresh Model 1 metrics:")
print({
    "accuracy": accuracy_score(ya_val, pred_fresh),
    "precision": precision_score(ya_val, pred_fresh, zero_division=0),
    "recall": recall_score(ya_val, pred_fresh, zero_division=0),
    "f1": f1_score(ya_val, pred_fresh, zero_division=0),
    "roc_auc": roc_auc_score(ya_val, proba_fresh),
    "pr_auc": average_precision_score(ya_val, proba_fresh),
    "pr_auc_baseline (no-skill)": baseline_prauc_1
})
print(classification_report(ya_val, pred_fresh, zero_division=0))

Model 1 class balance: neg=790, pos=86, scale_pos_weight=9.2
City overlap between train and val (should be 0): 0
Train: 715 rows / 46 cities, Val: 161 rows / 11 cities, val positive rate: 9.9%



Fresh Model 1 metrics:
{'accuracy': 0.8074534161490683, 'precision': 0.3170731707317073, 'recall': 0.8125, 'f1': 0.45614035087719296, 'roc_auc': 0.9034482758620689, 'pr_auc': 0.4249332365189563, 'pr_auc_baseline (no-skill)': np.float64(0.09937888198757763)}
              precision    recall  f1-score   support

           0       0.97      0.81      0.88       145
           1       0.32      0.81      0.46        16

    accuracy                           0.81       161
   macro avg       0.65      0.81      0.67       161
weighted avg       0.91      0.81      0.84       161



In [13]:
model1_metrics = pd.DataFrame([{
    "model": "Model 1 (explanatory, target=collapse_onset, at-risk population)",
    "n_training_cities": groups_m1.nunique(),
    "accuracy": accuracy_score(ya_val, pred_fresh),
    "precision": precision_score(ya_val, pred_fresh, zero_division=0),
    "recall": recall_score(ya_val, pred_fresh, zero_division=0),
    "f1": f1_score(ya_val, pred_fresh, zero_division=0),
    "roc_auc": roc_auc_score(ya_val, proba_fresh),
    "pr_auc": average_precision_score(ya_val, proba_fresh),
    "pr_auc_baseline": baseline_prauc_1
}])
model1_metrics.to_csv("output/tables/model1_metrics_final.csv", index=False)
print("Saved output/tables/model1_metrics_final.csv")

Saved output/tables/model1_metrics_final.csv


In [14]:
explainer = shap.TreeExplainer(model1_fresh)
shap_values = explainer.shap_values(Xa_train)

plt.figure()
shap.summary_plot(shap_values, Xa_train, show=False)
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model1.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved output/figures/shap_summary_model1.png")

shap_importance = pd.DataFrame({
    "feature": ALL_FEATURES,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
shap_importance.to_csv("output/tables/shap_importance_model1.csv", index=False)
print(shap_importance)

Saved output/figures/shap_summary_model1.png
                               feature  mean_abs_shap
0                  price_to_income_lag       1.198685
3                         zhvi_qoq_lag       0.744603
10               unemployment_rate_lag       0.625030
4   three-year_home_price_growth_trend       0.481822
2                         zhvi_yoy_lag       0.237311
9                         zori_yoy_lag       0.207360
12                       sp500_yoy_lag       0.205614
5                          hpi_yoy_lag       0.072410
11                         inv_qoq_lag       0.069472
7                     pop_velocity_lag       0.059446
8                 pop_acceleration_lag       0.037172
1              price_to_income_5yr_chg       0.032112
6                      hpi_3yr_chg_lag       0.028067


### Model 2: Generalization/Scoring Model

Same event-city training population and full feature set as Model 1. Optuna-tuned (including `scale_pos_weight` in the search space, since the right amount of imbalance correction isn't obvious a priori); its final fitted version is the one used to score the holdout set of at-risk, non-training metros.

In [15]:
neg2, pos2 = (yb_all == 0).sum(), (yb_all == 1).sum()
class_ratio_2 = neg2 / pos2
print(f"Model 2 class balance: neg={neg2}, pos={pos2}, class_ratio={class_ratio_2:.1f}")

split_m2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx_m2, val_idx_m2 = next(split_m2.split(Xb_all, yb_all, groups=groups_m2))
Xb_train, Xb_val = Xb_all.iloc[train_idx_m2], Xb_all.iloc[val_idx_m2]
yb_train, yb_val = yb_all.iloc[train_idx_m2], yb_all.iloc[val_idx_m2]
groups_train_m2 = groups_m2.iloc[train_idx_m2]

train_cities_m2 = set(groups_m2.iloc[train_idx_m2])
val_cities_m2 = set(groups_m2.iloc[val_idx_m2])
print("City overlap between train and val (should be 0):", len(train_cities_m2 & val_cities_m2))
print(f"Train: {len(Xb_train)} rows / {len(train_cities_m2)} cities, "
      f"Val: {len(Xb_val)} rows / {len(val_cities_m2)} cities, "
      f"val positive rate: {yb_val.mean():.1%}")

Model 2 class balance: neg=790, pos=86, class_ratio=9.2
City overlap between train and val (should be 0): 0
Train: 715 rows / 46 cities, Val: 161 rows / 11 cities, val positive rate: 9.9%


In [16]:
# Hyperparameter + threshold tuning for Model 2 (Optuna, 100 trials, grouped CV on PR-AUC).
# PR-AUC (not F1) is used as the Optuna objective, since F1 requires picking a
# threshold and can be unstable to optimize directly at this class balance;
# the threshold itself is picked separately below using out-of-fold predictions.
Path("output/tables").mkdir(parents=True, exist_ok=True)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)


def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 2, 4),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 2.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, class_ratio_2 * 1.5),
        "monotone_constraints": MONOTONE_INCREASING,
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1
    }
    return cross_val_score(
        xgb.XGBClassifier(**params), Xb_train, yb_train,
        groups=groups_train_m2, scoring="average_precision", cv=cv, n_jobs=-1
    ).mean()


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

best_params = {
    **study.best_params,
    "monotone_constraints": MONOTONE_INCREASING,
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1
}
print("\nBest parameters:", study.best_params)
print(f"Best cross-validation PR-AUC: {study.best_value:.3f} (baseline: {yb_train.mean():.3f})")

# Find the best classification cutoff using out-of-fold predictions
base_model = xgb.XGBClassifier(**best_params)
oof_probabilities = cross_val_predict(
    base_model, Xb_train, yb_train, groups=groups_train_m2, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

thresholds = np.arange(0.05, 0.96, 0.01)
f1_scores = [
    f1_score(yb_train, oof_probabilities >= threshold, zero_division=0)
    for threshold in thresholds
]
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"\nOptimal threshold: {best_threshold:.2f}")
print(f"Best OOF F1: {max(f1_scores):.3f}")

# Fit the final model and evaluate on untouched validation data
model2_final = xgb.XGBClassifier(**best_params)
model2_final.fit(Xb_train, yb_train)

validation_probabilities = model2_final.predict_proba(Xb_val)[:, 1]
validation_predictions = (validation_probabilities >= best_threshold).astype(int)

tuned_metrics = {
    "model": "Model 2 (generalization/scoring, target=collapse_onset, at-risk population)",
    "n_training_cities": groups_m2.nunique(),
    "features": ", ".join(ALL_FEATURES),
    "threshold": round(best_threshold, 2),
    "accuracy": accuracy_score(yb_val, validation_predictions),
    "precision": precision_score(yb_val, validation_predictions, zero_division=0),
    "recall": recall_score(yb_val, validation_predictions, zero_division=0),
    "f1": f1_score(yb_val, validation_predictions, zero_division=0),
    "roc_auc": roc_auc_score(yb_val, validation_probabilities),
    "pr_auc": average_precision_score(yb_val, validation_probabilities),
    "pr_auc_baseline": yb_val.mean()
}

print("\nTuned Model 2 validation metrics:")
for name, value in tuned_metrics.items():
    print(f"{name}: {value}")
print("\nClassification report:")
print(classification_report(yb_val, validation_predictions, zero_division=0))

[I 2026-08-31 19:44:32,066] A new study created in memory with name: no-name-0aa38752-b3f0-48ec-8ec9-35a44507c8bc


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-08-31 19:44:33,990] Trial 0 finished with value: 0.5706487691041611 and parameters: {'max_depth': 3, 'min_child_weight': 7, 'learning_rate': 0.07259248719561363, 'n_estimators': 140, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 3.0349658373387986e-08, 'reg_lambda': 0.6245760287469887, 'scale_pos_weight': 8.681690673323098}. Best is trial 0 with value: 0.5706487691041611.


[I 2026-08-31 19:44:35,249] Trial 1 finished with value: 0.5890386369205716 and parameters: {'max_depth': 4, 'min_child_weight': 1, 'learning_rate': 0.13826189316223855, 'n_estimators': 175, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'reg_alpha': 3.3300161336615e-07, 'reg_lambda': 5.472429642032189e-06, 'scale_pos_weight': 7.705899050742202}. Best is trial 1 with value: 0.5890386369205716.
[I 2026-08-31 19:44:35,296] Trial 2 finished with value: 0.5852351112679842 and parameters: {'max_depth': 3, 'min_child_weight': 3, 'learning_rate': 0.05243180891902853, 'n_estimators': 71, 'subsample': 0.7168578594140872, 'colsample_bytree': 0.7465447373174766, 'reg_alpha': 6.107319200689796e-05, 'reg_lambda': 0.11656915613247415, 'scale_pos_weight': 3.551645192930667}. Best is trial 1 with value: 0.5890386369205716.


[I 2026-08-31 19:44:36,454] Trial 3 finished with value: 0.5353550307806675 and parameters: {'max_depth': 3, 'min_child_weight': 5, 'learning_rate': 0.011340440501807348, 'n_estimators': 141, 'subsample': 0.6682096494749166, 'colsample_bytree': 0.6260206371941118, 'reg_alpha': 0.7528826814605758, 'reg_lambda': 4.905556676028766, 'scale_pos_weight': 11.330566111395243}. Best is trial 1 with value: 0.5890386369205716.


...

[I 2026-08-31 19:44:43,870] Trial 86 finished with value: 0.611845852242878 and parameters: {'max_depth': 4, 'min_child_weight': 2, 'learning_rate': 0.04780035230526813, 'n_estimators': 126, 'subsample': 0.994839482346132, 'colsample_bytree': 0.9960671881899769, 'reg_alpha': 5.98171032019442e-07, 'reg_lambda': 0.27293617231867934, 'scale_pos_weight': 12.88674232928096}. Best is trial 41 with value: 0.6374270489787619.
[I 2026-08-31 19:44:43,934] Trial 87 finished with value: 0.6329231702465256 and parameters: {'max_depth': 4, 'min_child_weight': 2, 'learning_rate': 0.040573269230605946, 'n_estimators': 117, 'subsample': 0.991183181020685, 'colsample_bytree': 0.9237186673155007, 'reg_alpha': 2.4953262161637615e-07, 'reg_lambda': 0.45299099871995013, 'scale_pos_weight': 10.20698899581601}. Best is trial 41 with value: 0.6374270489787619.
[I 2026-08-31 19:44:43,996] Trial 88 finished with value: 0.6054515431834553 and parameters: {'max_depth': 4, 'min_child_weight': 2, 'learning_rate': 0.

[I 2026-08-31 19:44:44,109] Trial 90 finished with value: 0.6102021953596211 and parameters: {'max_depth': 2, 'min_child_weight': 1, 'learning_rate': 0.0855034835654475, 'n_estimators': 116, 'subsample': 0.9919521257844106, 'colsample_bytree': 0.9540542415585821, 'reg_alpha': 1.3801130312532722e-07, 'reg_lambda': 3.1338988769158824, 'scale_pos_weight': 6.622765256743334}. Best is trial 41 with value: 0.6374270489787619.
[I 2026-08-31 19:44:44,181] Trial 91 finished with value: 0.6055159041888158 and parameters: {'max_depth': 4, 'min_child_weight': 2, 'learning_rate': 0.03590312647031226, 'n_estimators': 134, 'subsample': 0.959988953574539, 'colsample_bytree': 0.9162933557181281, 'reg_alpha': 0.002977245074028549, 'reg_lambda': 0.35680486272229495, 'scale_pos_weight': 10.26149125683635}. Best is trial 41 with value: 0.6374270489787619.
[I 2026-08-31 19:44:44,242] Trial 92 finished with value: 0.6073343799801472 and parameters: {'max_depth': 4, 'min_child_weight': 2, 'learning_rate': 0.0

[I 2026-08-31 19:44:44,315] Trial 93 finished with value: 0.6196794326436239 and parameters: {'max_depth': 4, 'min_child_weight': 2, 'learning_rate': 0.037541853886771656, 'n_estimators': 145, 'subsample': 0.9910591543576496, 'colsample_bytree': 0.9249425179126606, 'reg_alpha': 3.338295996130655e-07, 'reg_lambda': 0.09530607328477068, 'scale_pos_weight': 7.373599620401837}. Best is trial 41 with value: 0.6374270489787619.
[I 2026-08-31 19:44:44,388] Trial 94 finished with value: 0.6185618671462793 and parameters: {'max_depth': 4, 'min_child_weight': 2, 'learning_rate': 0.028640495735670343, 'n_estimators': 129, 'subsample': 0.9993350511000708, 'colsample_bytree': 0.8988417923186525, 'reg_alpha': 1.836128675592288e-07, 'reg_lambda': 0.5667490619632385, 'scale_pos_weight': 7.72773409921807}. Best is trial 41 with value: 0.6374270489787619.
[I 2026-08-31 19:44:44,448] Trial 95 finished with value: 0.610088938911199 and parameters: {'max_depth': 4, 'min_child_weight': 2, 'learning_rate': 0

[I 2026-08-31 19:44:44,572] Trial 97 finished with value: 0.5729913067440702 and parameters: {'max_depth': 3, 'min_child_weight': 1, 'learning_rate': 0.030198857989830644, 'n_estimators': 141, 'subsample': 0.956342883865309, 'colsample_bytree': 0.9316438067493731, 'reg_alpha': 1.0962924317149377e-07, 'reg_lambda': 0.26733671325030095, 'scale_pos_weight': 11.130140502955012}. Best is trial 41 with value: 0.6374270489787619.
[I 2026-08-31 19:44:44,642] Trial 98 finished with value: 0.6285614520008276 and parameters: {'max_depth': 4, 'min_child_weight': 5, 'learning_rate': 0.03770300515421204, 'n_estimators': 150, 'subsample': 0.9646974565894478, 'colsample_bytree': 0.919004576834832, 'reg_alpha': 3.308637492709557e-08, 'reg_lambda': 8.967400163064612e-05, 'scale_pos_weight': 12.387258103012519}. Best is trial 41 with value: 0.6374270489787619.
[I 2026-08-31 19:44:44,704] Trial 99 finished with value: 0.6104751298176659 and parameters: {'max_depth': 3, 'min_child_weight': 5, 'learning_rat


Optimal threshold: 0.56
Best OOF F1: 0.545

Tuned Model 2 validation metrics:
model: Model 2 (generalization/scoring, target=collapse_onset, at-risk population)
n_training_cities: 57
features: price_to_income_lag, price_to_income_5yr_chg, zhvi_yoy_lag, zhvi_qoq_lag, three-year_home_price_growth_trend, hpi_yoy_lag, hpi_3yr_chg_lag, pop_velocity_lag, pop_acceleration_lag, zori_yoy_lag, unemployment_rate_lag, inv_qoq_lag, sp500_yoy_lag
threshold: 0.56
accuracy: 0.8509316770186336
precision: 0.375
recall: 0.75
f1: 0.5
roc_auc: 0.9103448275862069
pr_auc: 0.4507360898905017
pr_auc_baseline: 0.09937888198757763

Classification report:
              precision    recall  f1-score   support

           0       0.97      0.86      0.91       145
           1       0.38      0.75      0.50        16

    accuracy                           0.85       161
   macro avg       0.67      0.81      0.71       161
weighted avg       0.91      0.85      0.87       161



In [17]:
model2_metrics = pd.DataFrame([tuned_metrics])
model2_metrics.to_csv("output/tables/model2_metrics_final.csv", index=False)
print("Saved output/tables/model2_metrics_final.csv")

Saved output/tables/model2_metrics_final.csv


## City holdout set

Score every at-risk metro outside the training population -- these are real candidates for early-warning risk ranking, since they're currently still affordable.

In [18]:
holdout_scoring = pd.read_csv("output/holdout_scoring.csv")
holdout_scoring["risk_score"] = model2_final.predict_proba(holdout_scoring[ALL_FEATURES])[:, 1]

city_risk = (
    holdout_scoring.groupby(["cbsa", "metro_name_x"])["risk_score"]
    .mean().reset_index().sort_values("risk_score", ascending=False)
)

city_risk.to_csv("output/tables/holdout_city_risk_scores.csv", index=False)
print("\nTop 15 highest-risk metros:")
print(city_risk.head(15))


Top 15 highest-risk metros:
      cbsa                        metro_name_x  risk_score
208  38340                      Pittsfield, MA    0.344070
75   20940                       El Centro, CA    0.315828
5    10740                     Albuquerque, NM    0.307984
206  38240        Pinehurst-Southern Pines, NC    0.302208
234  41940  San Jose-Sunnyvale-Santa Clara, CA    0.299765
49   16820                 Charlottesville, VA    0.279686
110  25260                Hanford-Corcoran, CA    0.266445
194  35380            New Orleans-Metairie, LA    0.236874
290  49340                       Worcester, MA    0.218979
142  28740                        Kingston, NY    0.212271
295  49740                            Yuma, AZ    0.195402
38   15260            Brunswick-St. Simons, GA    0.190133
71   20100                           Dover, DE    0.185569
212  39580                    Raleigh-Cary, NC    0.173184
143  28940                       Knoxville, TN    0.171558


In [19]:
top15 = city_risk.head(15).sort_values("risk_score", ascending=False)
plt.figure(figsize=(8, 6))
plt.barh(top15['metro_name_x'], top15['risk_score'], color="firebrick")
plt.xlabel("Risk Score")
plt.title("Top 15 Highest-Risk Metros (early-warning: probability of onset)")
plt.tight_layout()
plt.savefig("output/figures/top15_highest_risk_metros.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print("Saved output/figures/top15_highest_risk_metros.png")

Saved output/figures/top15_highest_risk_metros.png


/var/folders/8w/0g4j4t9j4jz3cfq3qwhrsn2r0000gn/T/ipykernel_45743/286874796.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Calibrating the holdout risk scores

Same two-part approach as before: a percentile rank (always safe) plus a Platt-scaled probability fit on out-of-fold predictions.

In [20]:
city_risk["risk_percentile"] = city_risk["risk_score"].rank(pct=True)

platt_scaler = LogisticRegression()
platt_scaler.fit(oof_probabilities.reshape(-1, 1), yb_train)

holdout_scoring["risk_score_calibrated"] = platt_scaler.predict_proba(
    holdout_scoring["risk_score"].values.reshape(-1, 1)
)[:, 1]

city_risk_calibrated = (
    holdout_scoring.groupby(["cbsa", "metro_name_x"])["risk_score_calibrated"]
    .mean().reset_index()
)
city_risk = city_risk.merge(city_risk_calibrated, on=["cbsa", "metro_name_x"], how="left")
city_risk.to_csv("output/tables/holdout_city_risk_scores.csv", index=False)

print(city_risk.sort_values("risk_score", ascending=False).head(15))

     cbsa                        metro_name_x  risk_score  risk_percentile  \
0   38340                      Pittsfield, MA    0.344070         1.000000   
1   20940                       El Centro, CA    0.315828         0.996622   
2   10740                     Albuquerque, NM    0.307984         0.993243   
3   38240        Pinehurst-Southern Pines, NC    0.302208         0.989865   
4   41940  San Jose-Sunnyvale-Santa Clara, CA    0.299765         0.986486   
5   16820                 Charlottesville, VA    0.279686         0.983108   
6   25260                Hanford-Corcoran, CA    0.266445         0.979730   
7   35380            New Orleans-Metairie, LA    0.236874         0.976351   
8   49340                       Worcester, MA    0.218979         0.972973   
9   28740                        Kingston, NY    0.212271         0.969595   
10  49740                            Yuma, AZ    0.195402         0.966216   
11  15260            Brunswick-St. Simons, GA    0.190133       

## Validation testing

Grouped cross-validation (`StratifiedGroupKFold`, cities never split across folds) across the full event-city population -- with ~55+ cities now instead of 3-20, this is a real cross-validation, not a token check.

In [21]:
n_splits_m1 = min(10, groups_m1.nunique())
group_cv = StratifiedGroupKFold(n_splits=n_splits_m1, shuffle=True, random_state=42)

cv_model1 = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
    scale_pos_weight=scale_pos_weight_1,
    monotone_constraints=MONOTONE_INCREASING, random_state=42
)
auc_scores_m1 = cross_val_score(cv_model1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring="roc_auc")
prauc_scores_m1 = cross_val_score(cv_model1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring="average_precision")

print(f"Model 1: grouped cross-validation ({n_splits_m1} folds)")
print(f"Mean AUC: {np.nanmean(auc_scores_m1):.3f} (std {np.nanstd(auc_scores_m1):.3f})")
print(f"Mean PR-AUC: {np.nanmean(prauc_scores_m1):.3f} (std {np.nanstd(prauc_scores_m1):.3f}) "
      f"-- baseline (positive rate): {y_all.mean():.3f}")

n_splits_m2 = min(10, groups_m2.nunique())
group_cv_m2 = StratifiedGroupKFold(n_splits=n_splits_m2, shuffle=True, random_state=42)
cv_model2 = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
    scale_pos_weight=class_ratio_2,
    monotone_constraints=MONOTONE_INCREASING, random_state=42
)
auc_scores_m2 = cross_val_score(cv_model2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring="roc_auc")
prauc_scores_m2 = cross_val_score(cv_model2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring="average_precision")

print(f"\nModel 2: grouped cross-validation ({n_splits_m2} folds)")
print(f"Mean AUC: {np.nanmean(auc_scores_m2):.3f} (std {np.nanstd(auc_scores_m2):.3f})")
print(f"Mean PR-AUC: {np.nanmean(prauc_scores_m2):.3f} (std {np.nanstd(prauc_scores_m2):.3f}) "
      f"-- baseline (positive rate): {yb_all.mean():.3f}")

cv_results = pd.DataFrame({
    "model": ["model 1"] * len(auc_scores_m1) + ["model 2"] * len(auc_scores_m2),
    "fold": list(range(1, len(auc_scores_m1) + 1)) + list(range(1, len(auc_scores_m2) + 1)),
    "roc_auc": list(auc_scores_m1) + list(auc_scores_m2),
    "pr_auc": list(prauc_scores_m1) + list(prauc_scores_m2),
})
cv_results.to_csv("output/tables/cv_results_final.csv", index=False)
print("\nSaved output/tables/cv_results_final.csv")

Model 1: grouped cross-validation (10 folds)
Mean AUC: 0.914 (std 0.045)
Mean PR-AUC: 0.587 (std 0.168) -- baseline (positive rate): 0.098



Model 2: grouped cross-validation (10 folds)
Mean AUC: 0.914 (std 0.045)
Mean PR-AUC: 0.587 (std 0.168) -- baseline (positive rate): 0.098

Saved output/tables/cv_results_final.csv


## Model comparison: logistic regression baseline

A second opinion alongside XGBoost: an L2-regularized logistic regression (features standardized first, `class_weight="balanced"`), evaluated with the same grouped CV.

In [22]:
logit_baseline_m1 = make_pipeline(
    StandardScaler(), LogisticRegression(penalty="l2", C=1.0, max_iter=1000, class_weight="balanced")
)
prauc_lr_m1 = cross_val_score(logit_baseline_m1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring="average_precision")
auc_lr_m1 = cross_val_score(logit_baseline_m1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring="roc_auc")

logit_baseline_m2 = make_pipeline(
    StandardScaler(), LogisticRegression(penalty="l2", C=1.0, max_iter=1000, class_weight="balanced")
)
prauc_lr_m2 = cross_val_score(logit_baseline_m2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring="average_precision")
auc_lr_m2 = cross_val_score(logit_baseline_m2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring="roc_auc")

comparison = pd.DataFrame([
    {"model": "Model 1 XGBoost", "mean_pr_auc": np.nanmean(prauc_scores_m1), "mean_auc": np.nanmean(auc_scores_m1)},
    {"model": "Model 1 Logistic Regression", "mean_pr_auc": np.nanmean(prauc_lr_m1), "mean_auc": np.nanmean(auc_lr_m1)},
    {"model": "Model 2 XGBoost", "mean_pr_auc": np.nanmean(prauc_scores_m2), "mean_auc": np.nanmean(auc_scores_m2)},
    {"model": "Model 2 Logistic Regression", "mean_pr_auc": np.nanmean(prauc_lr_m2), "mean_auc": np.nanmean(auc_lr_m2)},
])
comparison.to_csv("output/tables/model_comparison_logreg_vs_xgboost.csv", index=False)
print(comparison)

                         model  mean_pr_auc  mean_auc
0              Model 1 XGBoost     0.586776  0.914223
1  Model 1 Logistic Regression     0.397554  0.846482
2              Model 2 XGBoost     0.586776  0.914223
3  Model 2 Logistic Regression     0.397554  0.846482


## SHAP explainability (diagnostic: with vs. without the price-to-income level features)

Full-data SHAP for Model 2, plus the sanity check from the plan: does the model still show real signal without `price_to_income_lag`/`price_to_income_5yr_chg`, the features closest to being a persistence proxy?

In [23]:
# Model 2, full feature set
model2_full = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
    scale_pos_weight=class_ratio_2,
    monotone_constraints=MONOTONE_INCREASING, random_state=42
)
model2_full.fit(Xb_all, yb_all)

explainer2 = shap.TreeExplainer(model2_full)
shap_values2 = explainer2.shap_values(Xb_all)

plt.figure()
shap.summary_plot(shap_values2, Xb_all, plot_type="bar", show=False)
plt.title("Model 2 -- all features, target=collapse_onset")
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model2_full.png", dpi=150)
plt.close()

plt.figure()
shap.summary_plot(shap_values2, Xb_all, show=False)
plt.title("Model 2 -- all features, target=collapse_onset")
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model2_beeswarm_full.png", dpi=150)
plt.close()

shap_importance_full_m2 = pd.DataFrame({
    "feature": ALL_FEATURES,
    "mean_abs_shap": np.abs(shap_values2).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
shap_importance_full_m2.to_csv("output/tables/shap_importance_model2_full.csv", index=False)
print("Model 2 (all features) SHAP importance:")
print(shap_importance_full_m2)

Model 2 (all features) SHAP importance:
                               feature  mean_abs_shap
0                  price_to_income_lag       1.209089
3                         zhvi_qoq_lag       0.677479
10               unemployment_rate_lag       0.549084
4   three-year_home_price_growth_trend       0.490803
2                         zhvi_yoy_lag       0.280824
9                         zori_yoy_lag       0.279346
12                       sp500_yoy_lag       0.189230
7                     pop_velocity_lag       0.073597
8                 pop_acceleration_lag       0.062898
11                         inv_qoq_lag       0.059232
6                      hpi_3yr_chg_lag       0.025010
5                          hpi_yoy_lag       0.022818
1              price_to_income_5yr_chg       0.020027


In [24]:
# Diagnostic: drop the price-to-income level features and re-check grouped CV
# performance -- confirms whether the model still has real signal without its
# most persistence-adjacent inputs.
NO_LEVEL_FEATURES = [f for f in ALL_FEATURES if f not in ("price_to_income_lag", "price_to_income_5yr_chg")]

cv_model2_noleveL = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
    scale_pos_weight=class_ratio_2, random_state=42
)
auc_noleveL = cross_val_score(
    cv_model2_noleveL, Xb_all[NO_LEVEL_FEATURES], yb_all, groups=groups_m2, cv=group_cv_m2, scoring="roc_auc"
)
prauc_noleveL = cross_val_score(
    cv_model2_noleveL, Xb_all[NO_LEVEL_FEATURES], yb_all, groups=groups_m2, cv=group_cv_m2, scoring="average_precision"
)

diagnostic = pd.DataFrame([
    {"feature_set": "All 13 features", "mean_auc": np.nanmean(auc_scores_m2), "mean_pr_auc": np.nanmean(prauc_scores_m2)},
    {"feature_set": "Without price-to-income level features (11 left)", "mean_auc": np.nanmean(auc_noleveL), "mean_pr_auc": np.nanmean(prauc_noleveL)},
])
diagnostic.to_csv("output/tables/diagnostic_without_level_features.csv", index=False)
print(diagnostic)
print(f"\nBaseline PR-AUC (positive rate): {yb_all.mean():.3f}")

                                        feature_set  mean_auc  mean_pr_auc
0                                   All 13 features  0.914223     0.586776
1  Without price-to-income level features (11 left)  0.843812     0.543129

Baseline PR-AUC (positive rate): 0.098


## Backtesting

**Leave-one-city-out** logistic-regression backtest (statsmodels `Logit`, L1-regularized), across every training city -- reported as a summary distribution rather than one line per city, given there are 50+ of them now.

In [25]:
def leave_one_city_out_backtest(X_all_bt, y_all_bt, groups, label):
    aucs = {}
    for city_code in sorted(groups.unique()):
        train_mask = groups != city_code
        test_mask = groups == city_code
        y_train_bt, y_test_bt = y_all_bt[train_mask], y_all_bt[test_mask]

        if y_train_bt.nunique() < 2 or y_test_bt.nunique() < 2:
            continue

        X_train_bt = X_all_bt[train_mask].copy()
        X_train_bt.insert(0, "const", 1.0)
        X_test_bt = X_all_bt[test_mask].copy()
        X_test_bt.insert(0, "const", 1.0)

        result = Logit(y_train_bt, X_train_bt).fit_regularized(method="l1", alpha=1.0, disp=0)
        pred_probs = result.predict(X_test_bt)
        auc = roc_auc_score(y_test_bt, pred_probs)
        aucs[city_code] = auc

    if aucs:
        vals = np.array(list(aucs.values()))
        print(f"{label}: {len(aucs)} cities had both classes present in their held-out fold")
        print(f"  mean AUC: {vals.mean():.3f}, median: {np.median(vals):.3f}, "
              f"min: {vals.min():.3f}, max: {vals.max():.3f}")
        worst = sorted(aucs.items(), key=lambda kv: kv[1])[:5]
        best = sorted(aucs.items(), key=lambda kv: -kv[1])[:5]
        city_names = model2_pool.groupby("cbsa")["metro_name_x"].first()
        print("  worst 5:", [(city_names.get(c, c), round(a, 3)) for c, a in worst])
        print("  best 5:", [(city_names.get(c, c), round(a, 3)) for c, a in best])
    return aucs


print("Model 2 backtest")
auc2_by_city = leave_one_city_out_backtest(Xb_all, yb_all, groups_m2, "Model 2")

Model 2 backtest


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.

Model 2: 57 cities had both classes present in their held-out fold
  mean AUC: 0.805, median: 0.917, min: 0.000, max: 1.000
  worst 5: [('Springfield, MA', 0.0), ('Traverse City, MI', 0.0), ('Olympia-Lacey-Tumwater, WA', 0.25), ('Lake Havasu City-Kingman, AZ', 0.3), ('Las Vegas-Henderson-North Las Vegas, NV', 0.333)]
  best 5: [('Athens-Clarke County, GA', 1.0), ('Auburn-Opelika, AL', 1.0), ('Austin-Round Rock-San Marcos, TX', 1.0), ('Bridgeport-Stamford-Danbury, CT', 1.0), ('Charlotte-Concord-Gastonia, NC-SC', 1.0)]


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.